In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.0 MB/s eta 0:00:00


In [ ]:
!pip uninstall pandas -y
!pip install pandas


Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 38.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
prophet 1.2.2 requires pandas<3,>=1.0.4, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# TGN EPOCHS (20 EPOCHS)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import TemporalData
from torch_geometric.nn import TGNMemory, TransformerConv
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastAggregator,
    LastNeighborLoader,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

BATCH_SIZE = 200
MEMORY_DIM = 64
TIME_DIM = 64
EMBEDDING_DIM = 64
NUM_NEIGHBORS = 10
NUM_HEADS = 2
DROPOUT = 0.1
LEARNING_RATE = 0.0001
EPOCHS = 50
PATIENCE = 10

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

torch.manual_seed(42)
np.random.seed(42)


class EcommerceDataProcessor:
    def __init__(self, filepath):
        self.filepath = filepath
        self.num_users = 0
        self.num_items = 0
        self.num_nodes = 0

    def load_and_prepare(self):
        print("Loading data...")
        df = pd.read_csv(self.filepath)

        if 'u' in df.columns:
            user_col, item_col, ts_col, label_col = 'u', 'i', 'ts', 'label'
        else:
            user_col, item_col, ts_col, label_col = 'user_id', 'item_id', 'timestamp', 'label'

        df = df.sort_values(ts_col).reset_index(drop=True)

        unique_users = df[user_col].unique()
        unique_items = df[item_col].unique()

        user_map = {u: i for i, u in enumerate(unique_users)}
        item_map = {it: i + len(unique_users) for i, it in enumerate(unique_items)}

        self.num_users = len(unique_users)
        self.num_items = len(unique_items)
        self.num_nodes = self.num_users + self.num_items

        src = torch.tensor([user_map[u] for u in df[user_col]], dtype=torch.long)
        dst = torch.tensor([item_map[i] for i in df[item_col]], dtype=torch.long)

        timestamps = df[ts_col].values
        t_min = timestamps.min()
        t = torch.tensor(timestamps - t_min, dtype=torch.long)

        labels = torch.tensor(df[label_col].values, dtype=torch.long)

        label_onehot = F.one_hot(labels, num_classes=3).float()

        dt = pd.to_datetime(df[ts_col], unit='ms')
        hour_sin = np.sin(2 * np.pi * dt.dt.hour / 24)
        hour_cos = np.cos(2 * np.pi * dt.dt.hour / 24)
        dow_sin = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
        dow_cos = np.cos(2 * np.pi * dt.dt.dayofweek / 7)

        time_features = torch.tensor(
            np.column_stack([hour_sin, hour_cos, dow_sin, dow_cos]),
            dtype=torch.float
        )

        msg = time_features

        print(f"\nDataset Statistics:")
        print(f"  Total nodes: {self.num_nodes}")
        print(f"  - Users: {self.num_users}")
        print(f"  - Items: {self.num_items}")
        print(f"  Total edges: {len(df)}")
        print(f"  Edge feature dim: {msg.shape[1]}")
        print(f"  Time range: {t.min():.4f} to {t.max():.4f} (normalized)")
        print(f"\n  Label distribution:")
        for label_val, name in [(0, 'View'), (1, 'Cart'), (2, 'Purchase')]:
            count = (labels == label_val).sum().item()
            print(f"    {name}: {count} ({100*count/len(labels):.1f}%)")

        data = TemporalData(
            src=src,
            dst=dst,
            t=t,
            msg=msg,
            y=labels,
        )

        self.user_map = user_map
        self.item_map = item_map
        self.user_map_inv = {v: k for k, v in user_map.items()}
        self.item_map_inv = {v: k for k, v in item_map.items()}

        return data

class GraphAttentionEmbedding(nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        edge_dim = msg_dim + time_enc.out_channels
        self.conv = TransformerConv(
            in_channels,
            out_channels // NUM_HEADS,
            heads=NUM_HEADS,
            dropout=DROPOUT,
            edge_dim=edge_dim,
            concat=True,
        )

    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = last_update[edge_index[0]] - t
        rel_t_enc = self.time_enc(rel_t.to(x.dtype))

        edge_attr = torch.cat([rel_t_enc, msg], dim=-1)

        return self.conv(x, edge_index, edge_attr)


class LinkPredictor(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.lin_src = nn.Linear(in_channels, in_channels)
        self.lin_dst = nn.Linear(in_channels, in_channels)
        self.lin_final = nn.Sequential(
            nn.Linear(in_channels, in_channels),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(in_channels, 1),
        )

    def forward(self, z_src, z_dst):
        h = self.lin_src(z_src) + self.lin_dst(z_dst)
        h = h.relu()
        return self.lin_final(h).squeeze(-1)


class EdgeClassifier(nn.Module):
    def __init__(self, in_channels, num_classes=3):
        super().__init__()
        self.lin = nn.Sequential(
            nn.Linear(in_channels * 2, in_channels),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(in_channels, num_classes),
        )

    def forward(self, z_src, z_dst):
        h = torch.cat([z_src, z_dst], dim=-1)
        return self.lin(h)


class TGN(nn.Module):
    def __init__(self, num_nodes, msg_dim, memory_dim, time_dim, embedding_dim, num_classes=3):
        super().__init__()

        self.num_nodes = num_nodes
        self.memory_dim = memory_dim

        self.memory = TGNMemory(
            num_nodes,
            msg_dim,
            memory_dim,
            time_dim,
            message_module=IdentityMessage(msg_dim, memory_dim, time_dim),
            aggregator_module=LastAggregator(),
        )

        self.gnn = GraphAttentionEmbedding(
            in_channels=memory_dim,
            out_channels=embedding_dim,
            msg_dim=msg_dim,
            time_enc=self.memory.time_enc,
        )

        self.link_pred = LinkPredictor(embedding_dim)

        self.edge_classifier = EdgeClassifier(embedding_dim, num_classes)

    def forward(self, batch, n_id, msg, t, edge_index, last_update):
        memory, _ = self.memory(n_id)

        z = self.gnn(memory, last_update, edge_index, t, msg)

        return z

    def compute_link_prob(self, z, src_idx, dst_idx):
        return self.link_pred(z[src_idx], z[dst_idx])

    def compute_edge_class(self, z, src_idx, dst_idx):
        return self.edge_classifier(z[src_idx], z[dst_idx])

    def update_memory(self, src, dst, t, msg):
        self.memory.update_state(src, dst, t, msg)

    def reset_memory(self):
        self.memory.reset_state()

    def detach_memory(self):
        self.memory.detach()

class NegativeSampler:
    def __init__(self, src, dst, num_nodes_dst):
        self.src = src.numpy()
        self.dst = dst.numpy()
        self.num_nodes_dst_start = src.max().item() + 1  # Items start after users
        self.num_nodes_dst = num_nodes_dst

        self.pos_edges = {}
        for s, d in zip(self.src, self.dst):
            if s not in self.pos_edges:
                self.pos_edges[s] = set()
            self.pos_edges[s].add(d)

    def sample(self, src_batch, num_neg=1):
        src_batch = src_batch.numpy() if torch.is_tensor(src_batch) else src_batch
        neg_dst = []

        for s in src_batch:
            pos_set = self.pos_edges.get(s, set())
            neg_candidates = []
            while len(neg_candidates) < num_neg:
                candidate = np.random.randint(
                    self.num_nodes_dst_start,
                    self.num_nodes_dst_start + self.num_nodes_dst
                )
                if candidate not in pos_set:
                    neg_candidates.append(candidate)
            neg_dst.append(neg_candidates)

        return torch.tensor(neg_dst, dtype=torch.long)


def train_epoch(model, data, train_mask, neighbor_loader, neg_sampler, optimizer, criterion_link, criterion_edge):
    model.train()
    model.reset_memory()
    neighbor_loader.reset_state()

    total_loss = 0
    total_link_loss = 0
    total_edge_loss = 0
    num_batches = 0

    all_link_probs = []
    all_link_labels = []
    all_edge_preds = []
    all_edge_labels = []

    train_src = data.src[train_mask]
    train_dst = data.dst[train_mask]
    train_t = data.t[train_mask]
    train_msg = data.msg[train_mask]
    train_y = data.y[train_mask]

    for i in range(0, train_src.size(0), BATCH_SIZE):
        batch_src = train_src[i:i+BATCH_SIZE].to(DEVICE)
        batch_dst = train_dst[i:i+BATCH_SIZE].to(DEVICE)
        batch_t = train_t[i:i+BATCH_SIZE].to(DEVICE)
        batch_msg = train_msg[i:i+BATCH_SIZE].to(DEVICE)
        batch_y = train_y[i:i+BATCH_SIZE].to(DEVICE)

        neg_dst = neg_sampler.sample(batch_src.cpu(), num_neg=1).squeeze(-1).to(DEVICE)

        n_id, edge_index, e_id = neighbor_loader(torch.cat([batch_src, batch_dst, neg_dst]).unique())
        n_id = n_id.to(DEVICE)
        edge_index = edge_index.to(DEVICE)

        neighbor_t = data.t[e_id.cpu()].to(DEVICE)
        neighbor_msg = data.msg[e_id.cpu()].to(DEVICE)

        last_update = model.memory.last_update[n_id].to(DEVICE)

        optimizer.zero_grad()

        z = model(
            batch=(batch_src, batch_dst),
            n_id=n_id,
            msg=neighbor_msg,
            t=neighbor_t,
            edge_index=edge_index,
            last_update=last_update,
        )

        assoc = torch.empty(n_id.max() + 1, dtype=torch.long, device=DEVICE)
        assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)

        src_idx = assoc[batch_src]
        dst_idx = assoc[batch_dst]
        neg_idx = assoc[neg_dst]

        pos_out = model.compute_link_prob(z, src_idx, dst_idx)
        neg_out = model.compute_link_prob(z, src_idx, neg_idx)

        link_loss = criterion_link(pos_out, torch.ones_like(pos_out))
        link_loss += criterion_link(neg_out, torch.zeros_like(neg_out))

        edge_logits = model.compute_edge_class(z, src_idx, dst_idx)
        edge_loss = criterion_edge(edge_logits, batch_y)

        loss = link_loss + 1.0 * edge_loss

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            all_link_probs.append(torch.sigmoid(pos_out).cpu())
            all_link_probs.append(torch.sigmoid(neg_out).cpu())
            all_link_labels.append(torch.ones(pos_out.size(0)))
            all_link_labels.append(torch.zeros(neg_out.size(0)))
            all_edge_preds.append(edge_logits.argmax(dim=-1).cpu())
            all_edge_labels.append(batch_y.cpu())

        model.update_memory(batch_src, batch_dst, batch_t, batch_msg)
        model.detach_memory()

        neighbor_loader.insert(batch_src, batch_dst)

        total_loss += loss.item()
        total_link_loss += link_loss.item()
        total_edge_loss += edge_loss.item()
        num_batches += 1

    link_probs = torch.cat(all_link_probs).numpy()
    link_labels = torch.cat(all_link_labels).numpy()
    link_preds = (link_probs > 0.5).astype(int)
    link_acc = accuracy_score(link_labels, link_preds)

    edge_preds = torch.cat(all_edge_preds).numpy()
    edge_labels = torch.cat(all_edge_labels).numpy()
    edge_acc = accuracy_score(edge_labels, edge_preds)

    return {
        'loss': total_loss / num_batches,
        'link_loss': total_link_loss / num_batches,
        'edge_loss': total_edge_loss / num_batches,
        'link_acc': link_acc,
        'edge_acc': edge_acc
    }

@torch.no_grad()
@torch.no_grad()
def evaluate(model, data, eval_mask, train_mask, neighbor_loader, neg_sampler):
    model.eval()
    model.reset_memory()
    neighbor_loader.reset_state()

    total_link_loss = 0
    total_edge_loss = 0
    num_batches = 0

    train_src = data.src[train_mask]
    train_dst = data.dst[train_mask]
    train_t = data.t[train_mask]
    train_msg = data.msg[train_mask]

    for i in range(0, train_src.size(0), BATCH_SIZE):
        batch_src = train_src[i:i+BATCH_SIZE].to(DEVICE)
        batch_dst = train_dst[i:i+BATCH_SIZE].to(DEVICE)
        batch_t = train_t[i:i+BATCH_SIZE].to(DEVICE)
        batch_msg = train_msg[i:i+BATCH_SIZE].to(DEVICE)

        model.update_memory(batch_src, batch_dst, batch_t, batch_msg)
        neighbor_loader.insert(batch_src, batch_dst)

    eval_src = data.src[eval_mask]
    eval_dst = data.dst[eval_mask]
    eval_t = data.t[eval_mask]
    eval_msg = data.msg[eval_mask]
    eval_y = data.y[eval_mask]

    link_probs = []
    link_labels = []
    edge_preds = []
    edge_labels = []

    for i in range(0, eval_src.size(0), BATCH_SIZE):
        batch_src = eval_src[i:i+BATCH_SIZE].to(DEVICE)
        batch_dst = eval_dst[i:i+BATCH_SIZE].to(DEVICE)
        batch_t = eval_t[i:i+BATCH_SIZE].to(DEVICE)
        batch_msg = eval_msg[i:i+BATCH_SIZE].to(DEVICE)
        batch_y = eval_y[i:i+BATCH_SIZE]

        neg_dst = neg_sampler.sample(batch_src.cpu(), num_neg=1).squeeze(-1).to(DEVICE)

        n_id, edge_index, e_id = neighbor_loader(torch.cat([batch_src, batch_dst, neg_dst]).unique())
        n_id = n_id.to(DEVICE)
        edge_index = edge_index.to(DEVICE)

        neighbor_t = data.t[e_id.cpu()].to(DEVICE)
        neighbor_msg = data.msg[e_id.cpu()].to(DEVICE)
        last_update = model.memory.last_update[n_id].to(DEVICE)

        z = model(
            batch=(batch_src, batch_dst),
            n_id=n_id,
            msg=neighbor_msg,
            t=neighbor_t,
            edge_index=edge_index,
            last_update=last_update,
        )

        assoc = torch.empty(n_id.max() + 1, dtype=torch.long, device=DEVICE)
        assoc[n_id] = torch.arange(n_id.size(0), device=DEVICE)

        src_idx = assoc[batch_src]
        dst_idx = assoc[batch_dst]
        neg_idx = assoc[neg_dst]

        pos_logits = model.compute_link_prob(z, src_idx, dst_idx)
        neg_logits = model.compute_link_prob(z, src_idx, neg_idx)
        edge_logits = model.compute_edge_class(z, src_idx, dst_idx)

        batch_link_loss = F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
        batch_link_loss += F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
        batch_edge_loss = F.cross_entropy(edge_logits, batch_y.to(DEVICE))

        total_link_loss += batch_link_loss.item()
        total_edge_loss += batch_edge_loss.item()
        num_batches += 1

        pos_prob = torch.sigmoid(pos_logits).cpu()
        neg_prob = torch.sigmoid(neg_logits).cpu()

        link_probs.append(pos_prob)
        link_probs.append(neg_prob)
        link_labels.append(torch.ones(pos_prob.size(0)))
        link_labels.append(torch.zeros(neg_prob.size(0)))

        edge_preds.append(edge_logits.argmax(dim=-1).cpu())
        edge_labels.append(batch_y)

        model.update_memory(batch_src, batch_dst, batch_t, batch_msg)
        neighbor_loader.insert(batch_src, batch_dst)

    link_probs = torch.cat(link_probs).numpy()
    link_labels = torch.cat(link_labels).numpy()
    edge_preds = torch.cat(edge_preds).numpy()
    edge_labels = torch.cat(edge_labels).numpy()

    link_auc = roc_auc_score(link_labels, link_probs)
    link_ap = average_precision_score(link_labels, link_probs)
    edge_acc = accuracy_score(edge_labels, edge_preds)
    edge_f1 = f1_score(edge_labels, edge_preds, average='macro')

    return {
        'link_auc': link_auc,
        'link_ap': link_ap,
        'edge_acc': edge_acc,
        'edge_f1': edge_f1,
        'loss': (total_link_loss + total_edge_loss) / num_batches,
        'link_probs': link_probs,
        'link_labels': link_labels,
        'edge_preds': edge_preds,
        'edge_labels': edge_labels,
    }

def main():
    processor = EcommerceDataProcessor('/content/drive/MyDrive/ColabNotebooks/togo_ai_labs/subset_50k.csv')
    data = processor.load_and_prepare()

    num_edges = data.src.size(0)
    train_end = int(0.7 * num_edges)
    val_end = int(0.85 * num_edges)

    train_mask = torch.arange(0, train_end)
    val_mask = torch.arange(train_end, val_end)
    test_mask = torch.arange(val_end, num_edges)

    print(f"\nData Split:")
    print(f"  Train: {len(train_mask)} edges")
    print(f"  Val:   {len(val_mask)} edges")
    print(f"  Test:  {len(test_mask)} edges")

    neighbor_loader = LastNeighborLoader(
        processor.num_nodes,
        size=NUM_NEIGHBORS,
        device=DEVICE
    )

    neg_sampler = NegativeSampler(
        data.src[:train_end],
        data.dst[:train_end],
        processor.num_items
    )

    model = TGN(
        num_nodes=processor.num_nodes,
        msg_dim=data.msg.size(1),
        memory_dim=MEMORY_DIM,
        time_dim=TIME_DIM,
        embedding_dim=EMBEDDING_DIM,
        num_classes=3,
    ).to(DEVICE)

    print(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")

    criterion_link = nn.BCEWithLogitsLoss()

    train_labels = data.y[train_mask]
    class_counts = torch.bincount(train_labels, minlength=3).float()
    class_weights = len(train_labels) / (3 * class_counts)
    class_weights = class_weights.to(DEVICE)
    print(f"Class weights: {class_weights}")

    criterion_edge = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("\n" + "="*70)
    print("TRAINING")
    print("="*70)

    best_val_auc = 0
    patience_counter = 0
    history = {
        'train_loss': [],
        'train_link_acc': [],
        'train_edge_acc': [],
        'val_loss': [],
        'val_link_acc': [],
        'val_edge_acc': [],
        'val_auc': [],
        'val_ap': []
    }

    for epoch in range(1, EPOCHS + 1):
      train_results = train_epoch(
          model, data, train_mask, neighbor_loader, neg_sampler,
          optimizer, criterion_link, criterion_edge
      )

      val_results = evaluate(model, data, val_mask, train_mask, neighbor_loader, neg_sampler)

      val_link_preds = (val_results['link_probs'] > 0.5).astype(int)
      val_link_acc = accuracy_score(val_results['link_labels'], val_link_preds)

      history['train_loss'].append(train_results['loss'])
      history['train_link_acc'].append(train_results['link_acc'])
      history['train_edge_acc'].append(train_results['edge_acc'])
      history['val_loss'].append(val_results['loss'])
      history['val_link_acc'].append(val_link_acc)
      history['val_edge_acc'].append(val_results['edge_acc'])
      history['val_auc'].append(val_results['link_auc'])
      history['val_ap'].append(val_results['link_ap'])

      print(f"Epoch {epoch:02d} | "
            f"Train Loss: {train_results['loss']:.4f} | "
            f"Train Link Acc: {train_results['link_acc']:.4f} | "
            f"Train Edge Acc: {train_results['edge_acc']:.4f} | "
            f"Val Loss: {val_results['loss']:.4f} | "
            f"Val Link Acc: {val_link_acc:.4f} | "
            f"Val Edge Acc: {val_results['edge_acc']:.4f} | "
            f"Val AUC: {val_results['link_auc']:.4f}")

      if val_results['link_auc'] > best_val_auc:
          best_val_auc = val_results['link_auc']
          patience_counter = 0
          torch.save(model.state_dict(), 'best_tgn_model.pt')
      else:
          patience_counter += 1
          if patience_counter >= PATIENCE:
              print(f"\nEarly stopping at epoch {epoch}")
              break

    model.load_state_dict(torch.load('best_tgn_model.pt'))

    print("\n" + "="*70)
    print("TEST EVALUATION")
    print("="*70)

    test_results = evaluate(model, data, test_mask, train_mask, neighbor_loader, neg_sampler)

    print(f"\nLink Prediction:")
    print(f"  AUC: {test_results['link_auc']:.4f}")
    print(f"  AP:  {test_results['link_ap']:.4f}")

    print(f"\nEdge Classification (View/Cart/Purchase):")
    print(f"  Accuracy: {test_results['edge_acc']:.4f}")
    print(f"  Macro F1: {test_results['edge_f1']:.4f}")

    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(test_results['edge_labels'], test_results['edge_preds'])
    print(f"         Pred View  Cart  Purchase")
    for i, row_name in enumerate(['View', 'Cart', 'Purchase']):
        print(f"  {row_name:8s}  {cm[i, 0]:5d}  {cm[i, 1]:4d}  {cm[i, 2]:8d}")

    print(f"\nClassification Report:")
    print(classification_report(
        test_results['edge_labels'],
        test_results['edge_preds'],
        target_names=['View', 'Cart', 'Purchase']
    ))

    print("\n" + "="*70)
    print("SCORE DISTRIBUTION ANALYSIS")
    print("="*70)

    pos_scores = test_results['link_probs'][test_results['link_labels'] == 1]
    neg_scores = test_results['link_probs'][test_results['link_labels'] == 0]

    print(f"Positive links - Mean: {pos_scores.mean():.4f}, Std: {pos_scores.std():.4f}")
    print(f"Negative links - Mean: {neg_scores.mean():.4f}, Std: {neg_scores.std():.4f}")

    if pos_scores.std() > 0 and neg_scores.std() > 0:
        cohens_d = (pos_scores.mean() - neg_scores.mean()) / np.sqrt(
            (pos_scores.std()**2 + neg_scores.std()**2) / 2
        )
        print(f"Separation (Cohen's d): {cohens_d:.4f}")

    print(f"\nPrediction variance check:")
    print(f"  Positive score variance: {pos_scores.var():.6f}")
    print(f"  Negative score variance: {neg_scores.var():.6f}")
    if pos_scores.var() < 1e-6:
        print("WARNING: Very low variance in positive predictions!")
    else:
        print("Predictions have healthy variance")

    return model, processor, data, history, test_results


if __name__ == "__main__":
    model, processor, data, history, test_results = main()

Using device: cpu
Loading data...

Dataset Statistics:
  Total nodes: 43637
  - Users: 36102
  - Items: 7535
  Total edges: 50000
  Edge feature dim: 4
  Time range: 0.0000 to 7430003614.0000 (normalized)

  Label distribution:
    View: 48171 (96.3%)
    Cart: 1381 (2.8%)
    Purchase: 448 (0.9%)

Data Split:
  Train: 35000 edges
  Val:   7500 edges
  Test:  7500 edges

Model Parameters: 92,420
Class weights: tensor([ 0.3459, 12.0898, 37.6344])

TRAINING
Epoch 01 | Train Loss: 2.3899 | Train Link Acc: 0.7032 | Train Edge Acc: 0.7951 | Val Loss: 2.1954 | Val Link Acc: 0.7359 | Val Edge Acc: 0.8448 | Val AUC: 0.8673
Epoch 02 | Train Loss: 1.7683 | Train Link Acc: 0.8855 | Train Edge Acc: 0.7058 | Val Loss: 4.0905 | Val Link Acc: 0.7719 | Val Edge Acc: 0.3308 | Val AUC: 0.8429
Epoch 03 | Train Loss: 1.6126 | Train Link Acc: 0.8940 | Train Edge Acc: 0.6971 | Val Loss: 5.1717 | Val Link Acc: 0.7724 | Val Edge Acc: 0.3500 | Val AUC: 0.8401
Epoch 04 | Train Loss: 1.6078 | Train Link Acc: 0.8

# HYBRID APPROACH-JODIE + TEMPORAL LINK + EDGE + TEMPORAL SMOTE (20 EPOCHS)

In [ ]:
# =====================================================
# COMBINED JODIE + TEMPORAL LINK + EDGE + TEMPORAL SMOTE
# =====================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from tqdm import tqdm

# ================= CONFIG =================
EMBED_DIM = 64
BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

torch.manual_seed(42)
np.random.seed(42)

DATA_PATH = "/content/drive/MyDrive/ColabNotebooks/togo_ai_labs/subset_50k.csv"


# =====================================================
# TEMPORAL SMOTE (SAFE LIGHT VERSION)
# =====================================================
def temporal_smote_light(df, time_col="ts", label_col="label", minority_classes=[1,2], multiplier=1.0):

    print("\nApplying Temporal SMOTE Light...")

    new_rows = []

    for cls in minority_classes:
        cls_df = df[df[label_col] == cls]

        if len(cls_df) < 2:
            continue

        target = int(len(cls_df) * multiplier)

        for _ in range(target):
            a = cls_df.sample(1).iloc[0]
            b = cls_df.sample(1).iloc[0]

            alpha = np.random.rand()

            new_time = int(alpha * a[time_col] + (1 - alpha) * b[time_col])

            new_row = a.copy()
            new_row[time_col] = new_time

            new_rows.append(new_row)

    if len(new_rows) > 0:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    print("After Temporal SMOTE:")
    print(df[label_col].value_counts())

    return df


# =====================================================
# DATA LOADER
# =====================================================
class EcommerceDataset:
    def __init__(self, path):

        df = pd.read_csv(path)

        if "u" in df.columns:
            user_col, item_col, time_col, label_col = "u", "i", "ts", "label"
        else:
            user_col, item_col, time_col, label_col = "user_id", "item_id", "timestamp", "label"

        # Temporal SMOTE BEFORE sorting
        df = temporal_smote_light(df, time_col=time_col)

        df = df.sort_values(time_col).reset_index(drop=True)

        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()

        df["u"] = self.user_enc.fit_transform(df[user_col])
        df["i"] = self.item_enc.fit_transform(df[item_col])

        self.users = torch.tensor(df["u"].values, dtype=torch.long)
        self.items = torch.tensor(df["i"].values, dtype=torch.long)
        self.times = torch.tensor(df[time_col].values, dtype=torch.float)
        self.labels = torch.tensor(df[label_col].values, dtype=torch.long)

        self.times = self.times - self.times.min()

        self.num_users = df["u"].nunique()
        self.num_items = df["i"].nunique()

        print("\nFinal Label Distribution:")
        for c in [0,1,2]:
            print(f"Class {c}: {(self.labels == c).sum().item()}")


# =====================================================
# JODIE MODEL (WITH LINK + EDGE HEADS)
# =====================================================
class JODIE(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, num_classes=3):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)

        self.user_gru = nn.GRUCell(embed_dim + 1, embed_dim)
        self.item_gru = nn.GRUCell(embed_dim + 1, embed_dim)

        # Edge classifier
        self.edge_classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes)
        )

        # Link predictor
        self.link_pred = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, u, i, dt):

        u_emb = self.user_emb(u)
        i_emb = self.item_emb(i)

        dt = dt.unsqueeze(1)

        u_new = self.user_gru(torch.cat([i_emb, dt], dim=1), u_emb)
        i_new = self.item_gru(torch.cat([u_emb, dt], dim=1), i_emb)

        # Safe state update
        with torch.no_grad():
            self.user_emb.weight[u] = u_new
            self.item_emb.weight[i] = i_new

        return u_new, i_new

    def edge_logits(self, u_emb, i_emb):
        return self.edge_classifier(torch.cat([u_emb, i_emb], dim=1))

    def link_logits(self, u_emb, i_emb):
        return self.link_pred(torch.cat([u_emb, i_emb], dim=1)).squeeze()


# =====================================================
# TRAIN
# =====================================================
def train_epoch(model, data, optimizer, edge_loss_fn, link_loss_fn):

    model.train()
    total_loss = 0

    for i in tqdm(range(0, len(data.users), BATCH_SIZE)):

        u = data.users[i:i+BATCH_SIZE].to(DEVICE)
        it = data.items[i:i+BATCH_SIZE].to(DEVICE)
        t = data.times[i:i+BATCH_SIZE].to(DEVICE)
        y = data.labels[i:i+BATCH_SIZE].to(DEVICE)

        optimizer.zero_grad()

        u_emb, i_emb = model(u, it, t)

        # Edge loss
        edge_logits = model.edge_logits(u_emb, i_emb)
        edge_loss = edge_loss_fn(edge_logits, y)

        # Link loss
        pos_logits = model.link_logits(u_emb, i_emb)

        neg_items = torch.randint(0, data.num_items, (len(u),), device=DEVICE)
        neg_emb = model.item_emb(neg_items)
        neg_logits = model.link_logits(u_emb, neg_emb)

        link_loss = link_loss_fn(pos_logits, torch.ones_like(pos_logits))
        link_loss += link_loss_fn(neg_logits, torch.zeros_like(neg_logits))

        loss = edge_loss + link_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss


# =====================================================
# EVALUATE
# =====================================================
@torch.no_grad()
def evaluate(model, data):

    model.eval()

    preds, labels = [], []
    link_scores, link_labels = [], []

    for i in range(0, len(data.users), BATCH_SIZE):

        u = data.users[i:i+BATCH_SIZE].to(DEVICE)
        it = data.items[i:i+BATCH_SIZE].to(DEVICE)
        t = data.times[i:i+BATCH_SIZE].to(DEVICE)
        y = data.labels[i:i+BATCH_SIZE].to(DEVICE)

        u_emb, i_emb = model(u, it, t)

        edge_logits = model.edge_logits(u_emb, i_emb)
        preds.append(edge_logits.argmax(1).cpu())
        labels.append(y.cpu())

        pos_score = torch.sigmoid(model.link_logits(u_emb, i_emb)).cpu()

        neg_items = torch.randint(0, data.num_items, (len(u),), device=DEVICE)
        neg_emb = model.item_emb(neg_items)
        neg_score = torch.sigmoid(model.link_logits(u_emb, neg_emb)).cpu()

        link_scores.append(pos_score)
        link_scores.append(neg_score)
        link_labels.append(torch.ones(len(pos_score)))
        link_labels.append(torch.zeros(len(neg_score)))

    preds = torch.cat(preds)
    labels = torch.cat(labels)
    link_scores = torch.cat(link_scores).numpy()
    link_labels = torch.cat(link_labels).numpy()

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(link_labels, link_scores)
    ap = average_precision_score(link_labels, link_scores)
    cm = confusion_matrix(labels, preds)

    return acc, f1, auc, ap, cm, classification_report(
        labels, preds, target_names=["View","Cart","Purchase"]
    )


# =====================================================
# MAIN
# =====================================================
def main():

    data = EcommerceDataset(DATA_PATH)

    model = JODIE(data.num_users, data.num_items, EMBED_DIM).to(DEVICE)

    # Tempered class weights (VERY IMPORTANT)
    counts = torch.bincount(data.labels).float()
    weights = 1 / torch.sqrt(counts)
    weights = weights / weights.sum() * 3
    weights = weights.to(DEVICE)

    print("Tempered Weights:", weights)

    edge_loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    link_loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\nTRAINING\n" + "="*60)

    for epoch in range(1, EPOCHS+1):
        loss = train_epoch(model, data, optimizer, edge_loss_fn, link_loss_fn)
        acc, f1, auc, ap, _, _ = evaluate(model, data)

        print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Acc {acc:.4f} | MacroF1 {f1:.4f} | AUC {auc:.4f}")

    print("\nFINAL EVAL\n" + "="*60)

    acc, f1, auc, ap, cm, report = evaluate(model, data)

    print("\nLink Prediction")
    print("AUC:", auc)
    print("AP :", ap)

    print("\nEdge Classification")
    print("Accuracy:", acc)
    print("Macro F1:", f1)

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(report)


if __name__ == "__main__":
    main()


Using device: cpu

Applying Temporal SMOTE Light...
After Temporal SMOTE:
label
0    48171
1     2762
2      896
Name: count, dtype: int64

Final Label Distribution:
Class 0: 48171
Class 1: 2762
Class 2: 896
Tempered Weights: tensor([0.2398, 1.0016, 1.7586])

TRAINING


100%|██████████| 203/203 [00:15<00:00, 12.81it/s]


Epoch 01 | Loss 417.0588 | Acc 0.9294 | MacroF1 0.3211 | AUC 0.8190


100%|██████████| 203/203 [00:07<00:00, 26.06it/s]


Epoch 02 | Loss 386.1099 | Acc 0.9294 | MacroF1 0.3211 | AUC 0.8651


...

Epoch 18 | Loss 241.8375 | Acc 0.9358 | MacroF1 0.6914 | AUC 0.9616


100%|██████████| 203/203 [00:08<00:00, 24.96it/s]


Epoch 19 | Loss 241.0895 | Acc 0.9376 | MacroF1 0.6979 | AUC 0.9608


100%|██████████| 203/203 [00:08<00:00, 23.18it/s]


Epoch 20 | Loss 238.7138 | Acc 0.9379 | MacroF1 0.6989 | AUC 0.9581

FINAL EVAL

Link Prediction
AUC: 0.9592540986841342
AP : 0.9486273541624713

Edge Classification
Accuracy: 0.93785332535839
Macro F1: 0.6988668569180444

Confusion Matrix
[[45717  1438  1016]
 [   45  2144   573]
 [    6   143   747]]

Classification Report
              precision    recall  f1-score   support

        View       1.00      0.95      0.97     48171
        Cart       0.58      0.78      0.66      2762
    Purchase       0.32      0.83      0.46       896

    accuracy                           0.94     51829
   macro avg       0.63      0.85      0.70     51829
weighted avg       0.96      0.94      0.95     51829



# HYBRID-JODIE + TEMPORAL SMOTE + TEMPORAL ADAPTIVE FOCAL + LINK PRED (50 EPOCHS)

In [ ]:
# =====================================================
# COMBINED JODIE + TEMPORAL SMOTE + TEMPORAL ADAPTIVE FOCAL + LINK PRED
# =====================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from tqdm import tqdm

# ================= CONFIG =================
EMBED_DIM = 64
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

torch.manual_seed(42)
np.random.seed(42)

DATA_PATH = "/content/drive/MyDrive/ColabNotebooks/togo_ai_labs/subset_50k.csv"


# =====================================================
# TEMPORAL SMOTE (LIGHT SAFE VERSION)
# =====================================================
def temporal_smote_light(df, time_col="ts", label_col="label",
                         minority_classes=[1,2], multiplier=1.0):

    print("\nApplying Temporal SMOTE Light...")
    new_rows = []

    for cls in minority_classes:
        cls_df = df[df[label_col] == cls]

        if len(cls_df) < 2:
            continue

        target = int(len(cls_df) * multiplier)

        for _ in range(target):
            a = cls_df.sample(1).iloc[0]
            b = cls_df.sample(1).iloc[0]

            alpha = np.random.rand()
            new_time = int(alpha * a[time_col] + (1-alpha) * b[time_col])

            new_row = a.copy()
            new_row[time_col] = new_time
            new_rows.append(new_row)

    if len(new_rows) > 0:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    print("After Temporal SMOTE:")
    print(df[label_col].value_counts())

    return df


# =====================================================
# TEMPORAL ADAPTIVE FOCAL LOSS
# =====================================================
class TemporalAdaptiveFocalLoss(nn.Module):
    def __init__(self, num_classes=3, gamma=2.0, eps=1e-6):
        super().__init__()
        self.gamma = gamma
        self.num_classes = num_classes
        self.eps = eps

    def forward(self, logits, targets):

        ce = F.cross_entropy(logits, targets, reduction='none')

        probs = F.softmax(logits, dim=1)
        pt = probs[torch.arange(len(targets)), targets]

        focal = (1 - pt) ** self.gamma

        counts = torch.bincount(targets, minlength=self.num_classes).float().to(logits.device)
        rarity = 1.0 / torch.sqrt(counts + self.eps)
        rarity = rarity / rarity.sum() * self.num_classes

        sample_weights = rarity[targets]

        loss = sample_weights * focal * ce
        return loss.mean()


# =====================================================
# DATASET
# =====================================================
class EcommerceDataset:
    def __init__(self, path):

        df = pd.read_csv(path)

        if "u" in df.columns:
            user_col, item_col, time_col, label_col = "u", "i", "ts", "label"
        else:
            user_col, item_col, time_col, label_col = "user_id", "item_id", "timestamp", "label"

        df = temporal_smote_light(df, time_col=time_col)

        df = df.sort_values(time_col).reset_index(drop=True)

        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()

        df["u"] = self.user_enc.fit_transform(df[user_col])
        df["i"] = self.item_enc.fit_transform(df[item_col])

        self.users = torch.tensor(df["u"].values, dtype=torch.long)
        self.items = torch.tensor(df["i"].values, dtype=torch.long)
        self.times = torch.tensor(df[time_col].values, dtype=torch.float)
        self.labels = torch.tensor(df[label_col].values, dtype=torch.long)

        self.times = self.times - self.times.min()

        self.num_users = df["u"].nunique()
        self.num_items = df["i"].nunique()

        print("\nFinal Label Distribution:")
        for c in [0,1,2]:
            print(f"Class {c}: {(self.labels==c).sum().item()}")


# =====================================================
# JODIE MODEL (EDGE + LINK HEADS)
# =====================================================
class JODIE(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, num_classes=3):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)

        self.user_gru = nn.GRUCell(embed_dim + 1, embed_dim)
        self.item_gru = nn.GRUCell(embed_dim + 1, embed_dim)

        self.edge_classifier = nn.Sequential(
            nn.Linear(embed_dim*2, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes)
        )

        self.link_pred = nn.Sequential(
            nn.Linear(embed_dim*2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, u, i, dt):

        u_emb = self.user_emb(u)
        i_emb = self.item_emb(i)

        dt = dt.unsqueeze(1)

        u_new = self.user_gru(torch.cat([i_emb, dt], dim=1), u_emb)
        i_new = self.item_gru(torch.cat([u_emb, dt], dim=1), i_emb)

        with torch.no_grad():
            self.user_emb.weight[u] = u_new
            self.item_emb.weight[i] = i_new

        return u_new, i_new

    def edge_logits(self, u_emb, i_emb):
        return self.edge_classifier(torch.cat([u_emb, i_emb], dim=1))

    def link_logits(self, u_emb, i_emb):
        return self.link_pred(torch.cat([u_emb, i_emb], dim=1)).squeeze()


# =====================================================
# TRAIN
# =====================================================
def train_epoch(model, data, optimizer, edge_loss_fn, link_loss_fn):

    model.train()
    total_loss = 0

    for i in tqdm(range(0, len(data.users), BATCH_SIZE)):

        u = data.users[i:i+BATCH_SIZE].to(DEVICE)
        it = data.items[i:i+BATCH_SIZE].to(DEVICE)
        t = data.times[i:i+BATCH_SIZE].to(DEVICE)
        y = data.labels[i:i+BATCH_SIZE].to(DEVICE)

        optimizer.zero_grad()

        u_emb, i_emb = model(u, it, t)

        edge_logits = model.edge_logits(u_emb, i_emb)
        edge_loss = edge_loss_fn(edge_logits, y)

        pos_logits = model.link_logits(u_emb, i_emb)

        neg_items = torch.randint(0, data.num_items, (len(u),), device=DEVICE)
        neg_emb = model.item_emb(neg_items)
        neg_logits = model.link_logits(u_emb, neg_emb)

        link_loss = link_loss_fn(pos_logits, torch.ones_like(pos_logits))
        link_loss += link_loss_fn(neg_logits, torch.zeros_like(neg_logits))

        loss = edge_loss + link_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss


# =====================================================
# EVALUATE
# =====================================================
@torch.no_grad()
def evaluate(model, data):

    model.eval()

    preds, labels = [], []
    link_scores, link_labels = [], []

    for i in range(0, len(data.users), BATCH_SIZE):

        u = data.users[i:i+BATCH_SIZE].to(DEVICE)
        it = data.items[i:i+BATCH_SIZE].to(DEVICE)
        t = data.times[i:i+BATCH_SIZE].to(DEVICE)
        y = data.labels[i:i+BATCH_SIZE].to(DEVICE)

        u_emb, i_emb = model(u, it, t)

        edge_logits = model.edge_logits(u_emb, i_emb)
        preds.append(edge_logits.argmax(1).cpu())
        labels.append(y.cpu())

        pos_score = torch.sigmoid(model.link_logits(u_emb, i_emb)).cpu()

        neg_items = torch.randint(0, data.num_items, (len(u),), device=DEVICE)
        neg_emb = model.item_emb(neg_items)
        neg_score = torch.sigmoid(model.link_logits(u_emb, neg_emb)).cpu()

        link_scores.append(pos_score)
        link_scores.append(neg_score)
        link_labels.append(torch.ones(len(pos_score)))
        link_labels.append(torch.zeros(len(neg_score)))

    preds = torch.cat(preds)
    labels = torch.cat(labels)
    link_scores = torch.cat(link_scores).numpy()
    link_labels = torch.cat(link_labels).numpy()

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(link_labels, link_scores)
    ap = average_precision_score(link_labels, link_scores)
    cm = confusion_matrix(labels, preds)

    return acc, f1, auc, ap, cm, classification_report(
        labels, preds, target_names=["View","Cart","Purchase"]
    )


# =====================================================
# MAIN
# =====================================================
def main():

    data = EcommerceDataset(DATA_PATH)

    model = JODIE(data.num_users, data.num_items, EMBED_DIM).to(DEVICE)

    edge_loss_fn = TemporalAdaptiveFocalLoss(num_classes=3, gamma=2.0)
    link_loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\nTRAINING\n" + "="*60)

    for epoch in range(1, EPOCHS+1):
        loss = train_epoch(model, data, optimizer, edge_loss_fn, link_loss_fn)
        acc, f1, auc, ap, _, _ = evaluate(model, data)

        print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Acc {acc:.4f} | MacroF1 {f1:.4f} | AUC {auc:.4f}")

    print("\nFINAL EVAL\n" + "="*60)

    acc, f1, auc, ap, cm, report = evaluate(model, data)

    print("\nLink Prediction")
    print("AUC:", auc)
    print("AP :", ap)

    print("\nEdge Classification")
    print("Accuracy:", acc)
    print("Macro F1:", f1)

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(report)


if __name__ == "__main__":
    main()


Using device: cpu

Applying Temporal SMOTE Light...
After Temporal SMOTE:
label
0    48171
1     2762
2      896
Name: count, dtype: int64

Final Label Distribution:
Class 0: 48171
Class 1: 2762
Class 2: 896

TRAINING


100%|██████████| 203/203 [00:15<00:00, 13.00it/s]


Epoch 01 | Loss 231.7128 | Acc 0.9294 | MacroF1 0.3211 | AUC 0.8269


100%|██████████| 203/203 [00:09<00:00, 22.47it/s]


Epoch 02 | Loss 219.5116 | Acc 0.9285 | MacroF1 0.4657 | AUC 0.8712


...

Epoch 48 | Loss 65.9903 | Acc 0.9313 | MacroF1 0.6842 | AUC 0.9804


100%|██████████| 203/203 [00:08<00:00, 22.91it/s]


Epoch 49 | Loss 64.9243 | Acc 0.9312 | MacroF1 0.6851 | AUC 0.9830


100%|██████████| 203/203 [00:09<00:00, 22.25it/s]


Epoch 50 | Loss 63.8056 | Acc 0.9295 | MacroF1 0.6784 | AUC 0.9810

FINAL EVAL

Link Prediction
AUC: 0.9812178816625031
AP : 0.9758370336957077

Edge Classification
Accuracy: 0.9295375176059735
Macro F1: 0.6784348257637122

Confusion Matrix
[[45455  1661  1055]
 [   82  1850   830]
 [    2    22   872]]

Classification Report
              precision    recall  f1-score   support

        View       1.00      0.94      0.97     48171
        Cart       0.52      0.67      0.59      2762
    Purchase       0.32      0.97      0.48       896

    accuracy                           0.93     51829
   macro avg       0.61      0.86      0.68     51829
weighted avg       0.96      0.93      0.94     51829



# HYBRID-JODIE + TEMPORAL SMOTE + TEMPORAL ADAPTIVE FOCAL + LINK PRED (200 EPOCHS)

In [ ]:
# =====================================================
# COMBINED JODIE + TEMPORAL SMOTE + TEMPORAL ADAPTIVE FOCAL + LINK PRED
# =====================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
from tqdm import tqdm

# ================= CONFIG =================
EMBED_DIM = 64
BATCH_SIZE = 256
EPOCHS = 200
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

torch.manual_seed(42)
np.random.seed(42)

DATA_PATH = "/content/drive/MyDrive/ColabNotebooks/togo_ai_labs/subset_50k.csv"


# =====================================================
# TEMPORAL SMOTE (LIGHT SAFE VERSION)
# =====================================================
def temporal_smote_light(df, time_col="ts", label_col="label",
                         minority_classes=[1,2], multiplier=1.0):

    print("\nApplying Temporal SMOTE Light...")
    new_rows = []

    for cls in minority_classes:
        cls_df = df[df[label_col] == cls]

        if len(cls_df) < 2:
            continue

        target = int(len(cls_df) * multiplier)

        for _ in range(target):
            a = cls_df.sample(1).iloc[0]
            b = cls_df.sample(1).iloc[0]

            alpha = np.random.rand()
            new_time = int(alpha * a[time_col] + (1-alpha) * b[time_col])

            new_row = a.copy()
            new_row[time_col] = new_time
            new_rows.append(new_row)

    if len(new_rows) > 0:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    print("After Temporal SMOTE:")
    print(df[label_col].value_counts())

    return df


# =====================================================
# TEMPORAL ADAPTIVE FOCAL LOSS
# =====================================================
class TemporalAdaptiveFocalLoss(nn.Module):
    def __init__(self, num_classes=3, gamma=2.0, eps=1e-6):
        super().__init__()
        self.gamma = gamma
        self.num_classes = num_classes
        self.eps = eps

    def forward(self, logits, targets):

        ce = F.cross_entropy(logits, targets, reduction='none')

        probs = F.softmax(logits, dim=1)
        pt = probs[torch.arange(len(targets)), targets]

        focal = (1 - pt) ** self.gamma

        counts = torch.bincount(targets, minlength=self.num_classes).float().to(logits.device)
        rarity = 1.0 / torch.sqrt(counts + self.eps)
        rarity = rarity / rarity.sum() * self.num_classes

        sample_weights = rarity[targets]

        loss = sample_weights * focal * ce
        return loss.mean()


# =====================================================
# DATASET
# =====================================================
class EcommerceDataset:
    def __init__(self, path):

        df = pd.read_csv(path)

        if "u" in df.columns:
            user_col, item_col, time_col, label_col = "u", "i", "ts", "label"
        else:
            user_col, item_col, time_col, label_col = "user_id", "item_id", "timestamp", "label"

        df = temporal_smote_light(df, time_col=time_col)

        df = df.sort_values(time_col).reset_index(drop=True)

        self.user_enc = LabelEncoder()
        self.item_enc = LabelEncoder()

        df["u"] = self.user_enc.fit_transform(df[user_col])
        df["i"] = self.item_enc.fit_transform(df[item_col])

        self.users = torch.tensor(df["u"].values, dtype=torch.long)
        self.items = torch.tensor(df["i"].values, dtype=torch.long)
        self.times = torch.tensor(df[time_col].values, dtype=torch.float)
        self.labels = torch.tensor(df[label_col].values, dtype=torch.long)

        self.times = self.times - self.times.min()

        self.num_users = df["u"].nunique()
        self.num_items = df["i"].nunique()

        print("\nFinal Label Distribution:")
        for c in [0,1,2]:
            print(f"Class {c}: {(self.labels==c).sum().item()}")


# =====================================================
# JODIE MODEL (EDGE + LINK HEADS)
# =====================================================
class JODIE(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, num_classes=3):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)

        self.user_gru = nn.GRUCell(embed_dim + 1, embed_dim)
        self.item_gru = nn.GRUCell(embed_dim + 1, embed_dim)

        self.edge_classifier = nn.Sequential(
            nn.Linear(embed_dim*2, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(embed_dim, num_classes)
        )

        self.link_pred = nn.Sequential(
            nn.Linear(embed_dim*2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, u, i, dt):

        u_emb = self.user_emb(u)
        i_emb = self.item_emb(i)

        dt = dt.unsqueeze(1)

        u_new = self.user_gru(torch.cat([i_emb, dt], dim=1), u_emb)
        i_new = self.item_gru(torch.cat([u_emb, dt], dim=1), i_emb)

        with torch.no_grad():
            self.user_emb.weight[u] = u_new
            self.item_emb.weight[i] = i_new

        return u_new, i_new

    def edge_logits(self, u_emb, i_emb):
        return self.edge_classifier(torch.cat([u_emb, i_emb], dim=1))

    def link_logits(self, u_emb, i_emb):
        return self.link_pred(torch.cat([u_emb, i_emb], dim=1)).squeeze()


# =====================================================
# TRAIN
# =====================================================
def train_epoch(model, data, optimizer, edge_loss_fn, link_loss_fn):

    model.train()
    total_loss = 0

    for i in tqdm(range(0, len(data.users), BATCH_SIZE)):

        u = data.users[i:i+BATCH_SIZE].to(DEVICE)
        it = data.items[i:i+BATCH_SIZE].to(DEVICE)
        t = data.times[i:i+BATCH_SIZE].to(DEVICE)
        y = data.labels[i:i+BATCH_SIZE].to(DEVICE)

        optimizer.zero_grad()

        u_emb, i_emb = model(u, it, t)

        edge_logits = model.edge_logits(u_emb, i_emb)
        edge_loss = edge_loss_fn(edge_logits, y)

        pos_logits = model.link_logits(u_emb, i_emb)

        neg_items = torch.randint(0, data.num_items, (len(u),), device=DEVICE)
        neg_emb = model.item_emb(neg_items)
        neg_logits = model.link_logits(u_emb, neg_emb)

        link_loss = link_loss_fn(pos_logits, torch.ones_like(pos_logits))
        link_loss += link_loss_fn(neg_logits, torch.zeros_like(neg_logits))

        loss = edge_loss + link_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss


# =====================================================
# EVALUATE
# =====================================================
@torch.no_grad()
def evaluate(model, data):

    model.eval()

    preds, labels = [], []
    link_scores, link_labels = [], []

    for i in range(0, len(data.users), BATCH_SIZE):

        u = data.users[i:i+BATCH_SIZE].to(DEVICE)
        it = data.items[i:i+BATCH_SIZE].to(DEVICE)
        t = data.times[i:i+BATCH_SIZE].to(DEVICE)
        y = data.labels[i:i+BATCH_SIZE].to(DEVICE)

        u_emb, i_emb = model(u, it, t)

        edge_logits = model.edge_logits(u_emb, i_emb)
        preds.append(edge_logits.argmax(1).cpu())
        labels.append(y.cpu())

        pos_score = torch.sigmoid(model.link_logits(u_emb, i_emb)).cpu()

        neg_items = torch.randint(0, data.num_items, (len(u),), device=DEVICE)
        neg_emb = model.item_emb(neg_items)
        neg_score = torch.sigmoid(model.link_logits(u_emb, neg_emb)).cpu()

        link_scores.append(pos_score)
        link_scores.append(neg_score)
        link_labels.append(torch.ones(len(pos_score)))
        link_labels.append(torch.zeros(len(neg_score)))

    preds = torch.cat(preds)
    labels = torch.cat(labels)
    link_scores = torch.cat(link_scores).numpy()
    link_labels = torch.cat(link_labels).numpy()

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(link_labels, link_scores)
    ap = average_precision_score(link_labels, link_scores)
    cm = confusion_matrix(labels, preds)

    return acc, f1, auc, ap, cm, classification_report(
        labels, preds, target_names=["View","Cart","Purchase"]
    )


# =====================================================
# MAIN
# =====================================================
def main():

    data = EcommerceDataset(DATA_PATH)

    model = JODIE(data.num_users, data.num_items, EMBED_DIM).to(DEVICE)

    edge_loss_fn = TemporalAdaptiveFocalLoss(num_classes=3, gamma=2.0)
    link_loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\nTRAINING\n" + "="*60)

    for epoch in range(1, EPOCHS+1):
        loss = train_epoch(model, data, optimizer, edge_loss_fn, link_loss_fn)
        acc, f1, auc, ap, _, _ = evaluate(model, data)

        print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Acc {acc:.4f} | MacroF1 {f1:.4f} | AUC {auc:.4f}")

    print("\nFINAL EVAL\n" + "="*60)

    acc, f1, auc, ap, cm, report = evaluate(model, data)

    print("\nLink Prediction")
    print("AUC:", auc)
    print("AP :", ap)

    print("\nEdge Classification")
    print("Accuracy:", acc)
    print("Macro F1:", f1)

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(report)


if __name__ == "__main__":
    main()


Using device: cpu

Applying Temporal SMOTE Light...
After Temporal SMOTE:
label
0    48171
1     2762
2      896
Name: count, dtype: int64

Final Label Distribution:
Class 0: 48171
Class 1: 2762
Class 2: 896

TRAINING


100%|██████████| 203/203 [00:15<00:00, 13.27it/s]


Epoch 01 | Loss 231.7128 | Acc 0.9294 | MacroF1 0.3211 | AUC 0.8269


100%|██████████| 203/203 [00:09<00:00, 22.07it/s]


Epoch 02 | Loss 219.5116 | Acc 0.9285 | MacroF1 0.4657 | AUC 0.8712


...

Epoch 198 | Loss 16.9319 | Acc 0.9390 | MacroF1 0.7063 | AUC 0.9986


100%|██████████| 203/203 [00:08<00:00, 25.24it/s]


Epoch 199 | Loss 17.0300 | Acc 0.9384 | MacroF1 0.7071 | AUC 0.9985


100%|██████████| 203/203 [00:09<00:00, 21.37it/s]


Epoch 200 | Loss 17.5353 | Acc 0.9402 | MacroF1 0.7088 | AUC 0.9988

FINAL EVAL

Link Prediction
AUC: 0.9985416437634927
AP : 0.9981411382315805

Edge Classification
Accuracy: 0.9402265141137202
Macro F1: 0.7087916318356631

Confusion Matrix
[[45840  1498   833]
 [   46  2113   603]
 [    3   115   778]]

Classification Report
              precision    recall  f1-score   support

        View       1.00      0.95      0.97     48171
        Cart       0.57      0.77      0.65      2762
    Purchase       0.35      0.87      0.50       896

    accuracy                           0.94     51829
   macro avg       0.64      0.86      0.71     51829
weighted avg       0.96      0.94      0.95     51829

